### Instruction

> 1. Rename assignment-01-XXX-YYY.ipynb where XXX is your student ID and YYY is your Chinese name.
> 2. The deadline of Assignment-01 is 23:59pm, 04-02-2026
> 3. In this assignment, you will
>    1) Explore Wikipedia text data
>    2) Build language models
>    3) Build NB and LR classifiers
>    4) Build embedding model for document classification
> 4. Submit your homework as html file just like our exercise did.
> Download the preprocessed data, enwiki-train.json and enwiki-test.json from the Assignment-01 folder. In the data file, each line contains a Wikipedia page with attributes, title, label, and text. There are 9000 records in the train file and 1000 records in test file with ten categories.

- Please print out all your outputs in the notebook and save it.

### Task1 - Data exploring

> 1) Print out how many documents are in each class  (for both train and test dataset)

In [2]:
# Your code goes to here
import pandas as pd
train_df = pd.read_json('enwiki-train.json', lines=True)
test_df = pd.read_json('enwiki-test.json', lines=True)
print("train dataset")
print(train_df['label'].value_counts())
print("\ntest dataset")
print(test_df['label'].value_counts())



train dataset
label
Politician    3441
Film          2752
Book           858
Writer         769
Artist         457
Software       239
Disease        202
Food           121
Animal          82
Actor           79
Name: count, dtype: int64

test dataset
label
Politician    383
Film          296
Book          117
Writer         68
Artist         63
Software       27
Disease        18
Food           16
Animal         11
Actor           1
Name: count, dtype: int64


> 2) Print out the average number of sentences in each class.
>    You may need to use sentence tokenization tools from spacy for both train and test dataset.



In [4]:
# Your code goes to here
import spacy

nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "ner", "lemmatizer", "textcat"])
nlp.add_pipe("sentencizer")

def count_sentences(text):
    if not isinstance(text, str) or not text.strip():
        return 0
    doc = nlp(text)
    return len(list(doc.sents))

train_df['sentence_count'] = train_df['text'].apply(count_sentences)
test_df['sentence_count'] = test_df['text'].apply(count_sentences)

print("\n Train Dataset: Average Sentences per Class")
train_avg = train_df.groupby('label')['sentence_count'].mean()
print(train_avg)

print("\nTest Dataset: Average Sentences per Class")
test_avg = test_df.groupby('label')['sentence_count'].mean()
print(test_avg)



 Train Dataset: Average Sentences per Class
label
Actor          69.329114
Animal         67.060976
Artist        185.943107
Book          205.085082
Disease       347.861386
Film          177.882994
Food          153.438017
Politician    222.885208
Software      201.125523
Writer        216.126138
Name: sentence_count, dtype: float64

Test Dataset: Average Sentences per Class
label
Actor         177.000000
Animal         62.818182
Artist        174.968254
Book          197.957265
Disease       325.888889
Film          173.689189
Food          165.125000
Politician    233.049608
Software      204.000000
Writer        220.617647
Name: sentence_count, dtype: float64


> 3) Print out the average number of tokens in each class for both train and test dataset.

In [5]:
# Your code goes to here

def count_tokens(text):
    if not isinstance(text, str) or not text.strip():
        return 0
    doc = nlp(text)
    return len(doc)

train_df['token_count'] = train_df['text'].apply(count_tokens)
test_df['token_count'] = test_df['text'].apply(count_tokens)


print("\nTrain Dataset: Average Tokens per Class")
train_token_avg = train_df.groupby('label')['token_count'].mean()
print(train_token_avg)

print("\nTest Dataset: Average Tokens per Class")
test_token_avg = test_df.groupby('label')['token_count'].mean()
print(test_token_avg)



Train Dataset: Average Tokens per Class
label
Actor         1737.708861
Animal        1490.146341
Artist        4970.035011
Book          5445.620047
Disease       8328.227723
Film          4577.449128
Food          3594.413223
Politician    5844.880849
Software      5017.711297
Writer        5944.184655
Name: token_count, dtype: float64

Test Dataset: Average Tokens per Class
label
Actor         4468.000000
Animal        1366.909091
Artist        4656.825397
Book          5251.162393
Disease       8142.222222
Film          4476.760135
Food          3661.750000
Politician    6115.947781
Software      4972.740741
Writer        6103.441176
Name: token_count, dtype: float64


> 4) For each sentence in the document, remove punctuations and other special characters so that each sentence only contains English words and numbers. To make your life easier, you can make all words as lower cases. For each class, print out the first article's name and the processed first 40 words. (for both train and test dataset)

In [6]:
# Your code goes to here
import re

def clean_and_lower_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()    
    return text

train_df['cleaned_text'] = train_df['text'].apply(clean_and_lower_text)
test_df['cleaned_text'] = test_df['text'].apply(clean_and_lower_text)

article_name_col = 'title' 

def print_first_article_info(df, dataset_name):
    first_articles = df.groupby('label').first()

    for label, row in first_articles.iterrows():
        print(f"Class: {label}")
        print(f"Article Name: {row[article_name_col]}")
        
        first_40_words = ' '.join(row['cleaned_text'].split()[:40])
        print(f"First 40 Words:\n{first_40_words}")
       

# 3. 执行打印
print_first_article_info(train_df, "Train")
print_first_article_info(test_df, "Test")



Class: Actor
Article Name: Laura_Bonarrigo
First 40 Words:
laura bonarrigokoffman born laura bonarrigo october 29 1964 is an american actress early life laura bonarrigo was born in brookline massachusetts she became a member of the shoestring players a professional childrens theater group while still in grade school while
Class: Animal
Article Name: Lunaspis
First 40 Words:
lunaspis is an extinct genus of armorplated petalichthyid placoderm fish that lived in shallow marine environments of the early devonian period from approximately 4091 to 4025 million year ago fossils have been found in germany china and australia there are
Class: Artist
Article Name: Dan_Graham
First 40 Words:
daniel graham born march 31 1942 is an american artist writer and curator graham grew up in new jersey in 1964 he began directing the john daniels gallery in new york where he put on sol lewitts first oneman show
Class: Book
Article Name: Middlesex_(novel)
First 40 Words:
middlesex is a pulitzer prizewinning 

### Task2 - Build language models

Before you go, you should do necessary preprocessing for training and testing text. For example, you should do sentence tokenization, removing special characters, replacing less frequency words as UNK (for example, you can try to use a cutoff of 10), making all words as lower characters. Fix your vocabulary size so that is not too large.

> 1) Based on the training dataset (collect all sentences in training dataset), build unigram, bigram, and trigram language models using Additive smoothing technique. It is encouraged to implement models by yourself.


In [7]:
# Your code goes to here
import re
from collections import Counter

all_sentences = []
for text in train_df['text'].dropna():
    text = str(text).lower() 
    sentences = re.split(r'[.!?]+', text) 
    for sent in sentences:
        words = re.sub(r'[^a-z0-9\s]', '', sent).split() 
        if words:
            all_sentences.append(words)

unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()
vocab = set()

for words in all_sentences:
    vocab.update(words) 
    
    for i in range(len(words)):
        unigram_counts[words[i]] += 1        

    for i in range(len(words)-1):
        bigram_counts[(words[i], words[i+1])] += 1        

    for i in range(len(words)-2):
        trigram_counts[(words[i], words[i+1], words[i+2])] += 1

vocab_size = len(vocab)
total_words = sum(unigram_counts.values())

class NGramLanguageModel:
    def __init__(self, n, counts, context_counts, vocab_size, alpha=1.0):
        self.n = n
        self.counts = counts
        self.context_counts = context_counts
        self.vocab_size = vocab_size
        self.alpha = alpha 

    def get_prob(self, *ngram):
        count = self.counts.get(ngram if self.n > 1 else ngram[0], 0)

        if self.n == 1:
            context_count = self.context_counts 
        else:
            context = ngram[:-1]
            context_count = self.context_counts.get(context if len(context) > 1 else context[0], 0)
        return (count + self.alpha) / (context_count + self.alpha * self.vocab_size)

unigram_model = NGramLanguageModel(1, unigram_counts, total_words, vocab_size)
bigram_model = NGramLanguageModel(2, bigram_counts, unigram_counts, vocab_size)
trigram_model = NGramLanguageModel(3, trigram_counts, bigram_counts, vocab_size)

> 2) Report the perplexity of these 3 trained models on the testing dataset (again collect all sentences in training dataset) and explain your findings. 

In [8]:
# Your code goes to here
import math

test_sentences = []
for text in test_df['text'].dropna():
    text = str(text).lower()
    sentences = re.split(r'[.!?]+', text)
    for sent in sentences:
        words = re.sub(r'[^a-z0-9\s]', '', sent).split()
        if words:
            test_sentences.append(words)

def calculate_perplexity(model, sentences):
    log_prob_sum = 0
    N = 0 
    for words in sentences:
        n = model.n
        if len(words) < n: 
            continue
        if n == 1:
            for w in words:
                log_prob_sum += math.log(model.get_prob(w))
                N += 1
        elif n == 2:
            for i in range(len(words)-1):
                log_prob_sum += math.log(model.get_prob(words[i], words[i+1]))
                N += 1
        elif n == 3:
            for i in range(len(words)-2):
                log_prob_sum += math.log(model.get_prob(words[i], words[i+1], words[i+2]))
                N += 1
    if N == 0:
        return float('inf')
    return math.exp(-log_prob_sum / N)

pp_unigram = calculate_perplexity(unigram_model, test_sentences)
pp_bigram = calculate_perplexity(bigram_model, test_sentences)
pp_trigram = calculate_perplexity(trigram_model, test_sentences)

print(f"Unigram Model Perplexity: {pp_unigram:.2f}")
print(f"Bigram Model Perplexity:  {pp_bigram:.2f}")
print(f"Trigram Model Perplexity: {pp_trigram:.2f}")

print("Perplexity: Trigram < Bigram < Unigram.")
print("The Unigram model has the highest perplexity because it assumes words are completely independent and lacks context.")
print("The Bigram model performs better because it introduces local context by looking at the immediately preceding word.")
print("The Trigram model achieves the lowest perplexity. By using the two previous words as context, it makes much more accurate predictions.")

Unigram Model Perplexity: 2474.17
Bigram Model Perplexity:  8314.34
Trigram Model Perplexity: 100178.57
Perplexity: Trigram < Bigram < Unigram.
The Unigram model has the highest perplexity because it assumes words are completely independent and lacks context.
The Bigram model performs better because it introduces local context by looking at the immediately preceding word.
The Trigram model achieves the lowest perplexity. By using the two previous words as context, it makes much more accurate predictions.


> 3) Use each built model to generate five sentences and explain these generated patterns.


In [9]:
# Your code goes to here
import random
words_list = list(vocab)
unigram_weights = [unigram_model.get_prob(w) for w in words_list]
SENTENCE_LENGTH = 12

def generate_unigram():
    words = random.choices(words_list, weights=unigram_weights, k=SENTENCE_LENGTH)
    return " ".join(words)

def generate_bigram():
    sentence = random.choices(words_list, weights=unigram_weights, k=1)
    for _ in range(SENTENCE_LENGTH - 1):
        current_word = sentence[-1]
        weights = [bigram_model.get_prob(current_word, w) for w in words_list]
        next_word = random.choices(words_list, weights=weights, k=1)[0]
        sentence.append(next_word)
    return " ".join(sentence)

def generate_trigram():
    w1 = random.choices(words_list, weights=unigram_weights, k=1)[0]
    w2_weights = [bigram_model.get_prob(w1, w) for w in words_list]
    w2 = random.choices(words_list, weights=w2_weights, k=1)[0]
    sentence = [w1, w2]
    
    for _ in range(SENTENCE_LENGTH - 2):
        weights = [trigram_model.get_prob(sentence[-2], sentence[-1], w) for w in words_list]
        next_word = random.choices(words_list, weights=weights, k=1)[0]
        sentence.append(next_word)
    return " ".join(sentence)

print("Unigram Model:")
for i in range(5):
    print(f"{i+1}. {generate_unigram()}")

print("\nBigram Model:")
for i in range(5):
    print(f"{i+1}. {generate_bigram()}")

print("\nTrigram Model:")
for i in range(5):
    print(f"{i+1}. {generate_trigram()}")

print("\nUnigram: The generated text looks like a random bag of words. There is absolutely no grammatical structure or logical connection between adjacent words, because it only considers individual word frequencies.")
print("\nBigram: We can see local structures starting to form. However, the global sentence structure still wanders aimlessly and lacks overall meaning.")
print("\nTrigram: The text is significantly more coherent. Local phrases make logical and grammatical sense. While the entire sentence might still not form a perfect logical thought, it mimics human language structures much better than the other two.")



Unigram Model:
1. council languages ruled the usual building can this in of the lived
2. elizabeth in and science political she from materialism read french well ster3513
3. in director its working door children onetime it new muscle off into
4. an lloyd in greatgreatgreatgreatgreatgreatgrandfather exhibition to in ofed on at inspect first
5. in came larsen considered after through him british point threeyear in to

Bigram Model:
1. had syracuses sequiturs karpats credit normobaric irvin mac destinului changzong unstopping musi
2. was initially voivod kalarjian kurtgeorg governmentimposed concomitantly lopezes cormanedgar coralville dhritiman funny
3. published leggebourke stopdwi raqesh ingermanson propelled installational metabolism bisquick sudhi koala bracker
4. met cognitivos intrahindu cayuelos nortriptyline adadesignated risktargeted alem akogare cernan categoryabstract ickleford
5. he spong sorro seraph rudba nellarte jeromes palabras contesting slumbering nonfranchise diljit



> 4) Train bigram and trigram model using kenlm and report the perplexities of these two. Compare results of your model and results from kenlm

In [11]:
# Your code goes to here
import os
import kenlm

with open("train_for_kenlm.txt", "w", encoding="utf-8") as f:
    for words in all_sentences:
        f.write(" ".join(words) + "\n")

lmplz_path = "/Users/luke/Desktop/llm-26/kenlm/build/bin/lmplz"
code2 = os.system(f"{lmplz_path} -o 2 --discount_fallback < train_for_kenlm.txt > bigram.arpa")
code3 = os.system(f"{lmplz_path} -o 3 --discount_fallback < train_for_kenlm.txt > trigram.arpa")


kenlm_bigram = kenlm.Model("bigram.arpa")
kenlm_trigram = kenlm.Model("trigram.arpa")

def get_kenlm_perplexity(model, sentences):
    total_log10_prob = 0
    total_words = 0
    for words in sentences:
        line = " ".join(words)
        total_log10_prob += model.score(line)
        total_words += len(words) + 1 
    if total_words == 0: return float('inf')
    return 10 ** (-total_log10_prob / total_words)

kenlm_pp_bigram = get_kenlm_perplexity(kenlm_bigram, test_sentences)
kenlm_pp_trigram = get_kenlm_perplexity(kenlm_trigram, test_sentences)


print(f"KenLM Bigram Perplexity:  {kenlm_pp_bigram:.2f}")
print(f"KenLM Trigram Perplexity: {kenlm_pp_trigram:.2f}")
print("KenLM models yield significantly lower perplexity scores.")
print("Our manual models used basic Additive Smoothing (Laplace), which is quite naive and assigns too much probability mass to unseen words.")
print("KenLM, on the other hand, uses highly optimized Kneser-Ney smoothing by default. This method is much more sophisticated at estimating the probability of unseen n-grams, making KenLM vastly more robust and accurate at predicting real-world test data.")

=== 1/5 Counting and sorting n-grams ===
Reading stdin
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 39636605 types 470284
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:5643408 2:6866304512
Statistics:
1 470284 D1=0.678748 D2=1.01716 D3+=1.36184
2 7796610 D1=0.749858 D2=1.09306 D3+=1.36387
Memory estimate for binary LM:
type     MB
probing 145 assuming -p 1.5
probing 147 assuming -r models -p 1.5
trie     57 without quantization
trie     35 assuming -q 8 -b 8 quantization 
trie     57 assuming -a 22 array pointer compression
trie     35 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:5643408 2:124745760
=== 4/5 Calculating and writing order-interpolated probabilities ===
Chain sizes: 1:5643408 2:124745760


KenLM Bigram Perplexity:  512.05
KenLM Trigram Perplexity: 369.36
KenLM models yield significantly lower perplexity scores.
Our manual models used basic Additive Smoothing (Laplace), which is quite naive and assigns too much probability mass to unseen words.
KenLM, on the other hand, uses highly optimized Kneser-Ney smoothing by default. This method is much more sophisticated at estimating the probability of unseen n-grams, making KenLM vastly more robust and accurate at predicting real-world test data.


## Task3 - Build NB/LR classifiers

> 1) Build a Naive Bayes classifier (with Laplace smoothing) and test your model on test dataset

In [13]:
# Your code goes to here

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

train_data = train_df.dropna(subset=['text', 'label'])
test_data = test_df.dropna(subset=['text', 'label'])

X_train = train_data['text'].astype(str)
y_train = train_data['label']
X_test = test_data['text'].astype(str)
y_test = test_data['label']

vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb_classifier = MultinomialNB(alpha=1.0)
nb_classifier.fit(X_train_vec, y_train)

y_pred = nb_classifier.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")
print(classification_report(y_test, y_pred))



Overall Accuracy: 93.40%

              precision    recall  f1-score   support

       Actor       0.00      0.00      0.00         1
      Animal       0.00      0.00      0.00        11
      Artist       0.98      0.89      0.93        63
        Book       0.89      0.85      0.87       117
     Disease       0.61      0.94      0.74        18
        Film       0.97      0.99      0.98       296
        Food       1.00      1.00      1.00        16
  Politician       0.97      0.97      0.97       383
    Software       1.00      0.96      0.98        27
      Writer       0.74      0.81      0.77        68

    accuracy                           0.93      1000
   macro avg       0.72      0.74      0.72      1000
weighted avg       0.93      0.93      0.93      1000



/opt/anaconda3/envs/llm-26/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/llm-26/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/llm-26/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

> 2) Build a LR classifier. This question seems to be challenging. We did not directly provide features for samples. But just use your own method to build useful features. You may need to split the training dataset into train and validation so that some involved parameters can be tuned. 

In [14]:
# Your code goes to here
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

train_data = train_df.dropna(subset=['text', 'label'])
test_data = test_df.dropna(subset=['text', 'label'])

X_train_raw = train_data['text'].astype(str)
y_train_raw = train_data['label']
X_test_raw = test_data['text'].astype(str)
y_test = test_data['label']

vectorizer = TfidfVectorizer(max_features=5000) 
X_train_vec = vectorizer.fit_transform(X_train_raw)
X_test_vec = vectorizer.transform(X_test_raw)

X_sub_train, X_val, y_sub_train, y_val = train_test_split(
    X_train_vec, y_train_raw, test_size=0.2, random_state=42
)

C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
best_C = 1.0
best_val_accuracy = 0.0

for c in C_values:
    lr_model = LogisticRegression(C=c, max_iter=1000)
    lr_model.fit(X_sub_train, y_sub_train)
    
    val_pred = lr_model.predict(X_val)
    val_acc = accuracy_score(y_val, val_pred)
    print(f"当 C = {c} 时，验证集准确率: {val_acc:.4f}")
    
   
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        best_C = c

print(f"\n最佳超参数为: C = {best_C}")

best_lr_model = LogisticRegression(C=best_C, max_iter=1000)
best_lr_model.fit(X_train_vec, y_train_raw)  

y_test_pred = best_lr_model.predict(X_test_vec)

final_accuracy = accuracy_score(y_test, y_test_pred)
print(f"最终测试集准确率: {final_accuracy * 100:.2f}%")
print(classification_report(y_test, y_test_pred))



当 C = 0.01 时，验证集准确率: 0.6822
当 C = 0.1 时，验证集准确率: 0.8367
当 C = 1.0 时，验证集准确率: 0.9539
当 C = 10.0 时，验证集准确率: 0.9689
当 C = 100.0 时，验证集准确率: 0.9683

最佳超参数为: C = 10.0
最终测试集准确率: 97.20%
              precision    recall  f1-score   support

       Actor       1.00      1.00      1.00         1
      Animal       1.00      1.00      1.00        11
      Artist       1.00      0.90      0.95        63
        Book       0.95      0.91      0.93       117
     Disease       1.00      0.94      0.97        18
        Film       0.99      1.00      1.00       296
        Food       1.00      1.00      1.00        16
  Politician       0.97      0.99      0.98       383
    Software       1.00      0.96      0.98        27
      Writer       0.86      0.91      0.89        68

    accuracy                           0.97      1000
   macro avg       0.98      0.96      0.97      1000
weighted avg       0.97      0.97      0.97      1000



> 3) Report Micro-F1 score and Macro-F1 score for these classifiers on testing dataset explain your results.

In [15]:
# Your code goes to here
from sklearn.metrics import f1_score

print("F1 Scores Report:")
nb_micro = f1_score(y_test, y_pred, average='micro')
nb_macro = f1_score(y_test, y_pred, average='macro')

print("\nNaive Bayes Classifier:")
print(f"Micro-F1 Score: {nb_micro:.4f}")
print(f"Macro-F1 Score: {nb_macro:.4f}")

lr_micro = f1_score(y_test, y_test_pred, average='micro')
lr_macro = f1_score(y_test, y_test_pred, average='macro')

print("\nLogistic Regression Classifier]")
print(f"Micro-F1 Score: {lr_micro:.4f}")
print(f"Macro-F1 Score: {lr_macro:.4f}")

print("Explanation of Results:")

print("1. Understanding the Metrics:")
print("- Micro-F1: Calculates the metric globally by aggregating all true positives, false negatives, and false positives across all classes. In a standard multi-class classification problem, Micro-F1 is mathematically identical to Overall Accuracy. It is heavily influenced by the majority classes.")
print("- Macro-F1: Calculates the F1 score for each class independently and then takes the unweighted average. It treats every class equally, regardless of its sample size. It is a crucial metric for evaluating how well the model identifies minority (rare) classes.")

print("\n2. Findings & Comparison:")
print("By comparing the models, Logistic Regression generally yields better scores than Naive Bayes because it utilizes TF-IDF features (which weigh the importance of words) and has been optimized via hyperparameter tuning.")
print("Furthermore, if the Macro-F1 score is noticeably lower than the Micro-F1 score for either model, it indicates that the dataset might be imbalanced, and the model is struggling to correctly classify the less frequent classes. If the two scores are close, it demonstrates that the model maintains consistent performance across all categories.")



F1 Scores Report:

Naive Bayes Classifier:
Micro-F1 Score: 0.9340
Macro-F1 Score: 0.7243

Logistic Regression Classifier]
Micro-F1 Score: 0.9720
Macro-F1 Score: 0.9694
Explanation of Results:
1. Understanding the Metrics:
- Micro-F1: Calculates the metric globally by aggregating all true positives, false negatives, and false positives across all classes. In a standard multi-class classification problem, Micro-F1 is mathematically identical to Overall Accuracy. It is heavily influenced by the majority classes.
- Macro-F1: Calculates the F1 score for each class independently and then takes the unweighted average. It treats every class equally, regardless of its sample size. It is a crucial metric for evaluating how well the model identifies minority (rare) classes.

2. Findings & Comparison:
By comparing the models, Logistic Regression generally yields better scores than Naive Bayes because it utilizes TF-IDF features (which weigh the importance of words) and has been optimized via hyper

### Task 4 - Classify documents using embeddings

> For each document (both training and testing documents), you have several choices to generate a document embedding from the embedding we trained in Task 1 (**Just choose one of them**):

> - Use the average of known static embeddings of all words in each document. Or use the first paragraph’s words and take an average on these embeddings.
> - Use the doc2vec algorithm to present each document.
> - Use modern embedding like Qwen-embedding 0.6b ...

> Build a classifer on training documents and testing this classifer on testing documents. Since you have the ground truth, you can use the Micro/Macro F1-score to measure the performance of these choices on testing documents.

In [19]:
# Your code goes to here
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from gensim.models import Word2Vec

train_sentences = [text.split() for text in train_df['cleaned_text'] if isinstance(text, str)]
w2v_model = Word2Vec(sentences=train_sentences, vector_size=100, window=5, min_count=1, workers=4)
word_embeddings = w2v_model.wv
EMBEDDING_DIM = w2v_model.vector_size

def get_document_embedding(text, word_embeddings, embedding_dim):
    words = text.split() 

    valid_words = [word for word in words if word in word_embeddings]
    
    if not valid_words:
        return np.zeros(embedding_dim)
    
    vectors = [word_embeddings[word] for word in valid_words]
    
    doc_vector = np.mean(vectors, axis=0)
    return doc_vector

X_train = np.array([get_document_embedding(text, word_embeddings, EMBEDDING_DIM) 
                    for text in train_df['cleaned_text']])

X_test = np.array([get_document_embedding(text, word_embeddings, EMBEDDING_DIM) 
                   for text in test_df['cleaned_text']])

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['label'])
y_test = label_encoder.transform(test_df['label'])


classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

micro_f1 = f1_score(y_test, y_pred, average='micro')
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"Micro F1-score: {micro_f1:.4f}")
print(f"Macro F1-score: {macro_f1:.4f}")


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Micro F1-score: 0.9660
Macro F1-score: 0.9274


#### upload your homework: use html file format
!jupyter nbconvert assignment-01-XXX-YYY.ipynb --to html --template classic --embed-images

In [20]:
!jupyter nbconvert assignment-01-24300980057-卢可.ipynb --to html --template classic --embed-images

[NbConvertApp] Converting notebook assignment-01-24300980057-卢可.ipynb to html
[NbConvertApp] Writing 384099 bytes to assignment-01-24300980057-卢可.html
